1. 흥행 등급 클러스터링 (Clustering Success Tiers)

In [18]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from IPython.display import display

# 1. 데이터 로드 및 중복 제거
df_sample = pd.read_csv('steam_stratified_sample.csv')
df_sample.columns = df_sample.columns.str.strip()
df_sample['appid'] = df_sample['appid'].astype(str)
df_sample = df_sample.drop_duplicates(subset=['appid'])

# 2. 유저님 제공 로직: 74개 필터링 (stratum 컬럼 사용)
target_strata = ['large_high', 'mid_high', 'small_high']
df_high = df_sample[df_sample['stratum'].isin(target_strata)].copy()

# 3. 흥행 등급 클러스터링 (K-Means)
target_data = df_high[['appid', 'total_reviews']].dropna().copy()

# 데이터 쏠림 방지를 위한 로그 변환
target_data['log_reviews'] = np.log1p(target_data['total_reviews'])

# AI 클러스터링 실행 (4개 그룹)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
target_data['cluster'] = kmeans.fit_predict(target_data[['log_reviews']])

# 4. 결과 정리 및 정렬
cluster_stats = target_data.groupby('cluster')['total_reviews'].agg(['min', 'max', 'count', 'median']).reset_index()
cluster_stats = cluster_stats.sort_values(by='min', ascending=False).reset_index(drop=True)

# 등급 이름 부여
cluster_stats['등급'] = ['Tier 1 (메가 히트)', 'Tier 2 (중박/안정권)', 'Tier 3 (본전/니치)', 'Tier 4 (위험/정체)']

# 보고서용 포맷팅
cluster_stats['리뷰 수 구간 (Threshold)'] = cluster_stats.apply(lambda x: f"{int(x['min']):,}개 ~ {int(x['max']):,}개", axis=1)
cluster_stats['중앙값 (해당 티어 평균)'] = cluster_stats['median'].apply(lambda x: f"{int(x):,}개")
cluster_stats['해당 게임 수'] = cluster_stats['count']
final_tier_report = cluster_stats[['등급', '리뷰 수 구간 (Threshold)', '중앙값 (해당 티어 평균)', '해당 게임 수']]

print("="*85)
print(" [최종 타겟] 'High' 게임 74개 기준, 4단계 성공 티어표")
print("="*85)
display(final_tier_report)
print("="*85)

 [최종 타겟] 'High' 게임 74개 기준, 4단계 성공 티어표


,등급,리뷰 수 구간 (Threshold),중앙값 (해당 티어 평균),해당 게임 수
0,Tier 1 (메가 히트),"65,046개 ~ 153,566개","111,087개",3
1,Tier 2 (중박/안정권),"7,532개 ~ 22,456개","11,061개",11
2,Tier 3 (본전/니치),"1,710개 ~ 6,132개","3,997개",19
3,Tier 4 (위험/정체),"383개 ~ 1,245개",534개,41


In [20]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from IPython.display import display

# 1. 데이터 로드 및 74개 정예 필터링
df_sample = pd.read_csv('steam_stratified_sample.csv')
df_sample.columns = df_sample.columns.str.strip()
df_sample['appid'] = df_sample['appid'].astype(str)
df_sample = df_sample.drop_duplicates(subset=['appid'])

# 유저님 제공 로직: stratum 컬럼과 정확한 그룹명 사용
target_strata = ['large_high', 'mid_high', 'small_high']
df_high = df_sample[df_sample['stratum'].isin(target_strata)].copy()

# 2. 체급 내 4단계 클러스터링 함수
def get_4_tiers_within_stratum(sub_df):
    if len(sub_df) < 4: # 게임이 4개보다 적으면 나눌 수 없음
        return ["분석불가"] * len(sub_df)
    
    # 로그 변환 후 K-Means (4개 그룹)
    log_vals = np.log1p(sub_df['total_reviews']).values.reshape(-1, 1)
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(log_vals)
    
    # 리뷰 수가 높은 순서대로 1, 2, 3, 4등급 부여
    centers = kmeans.cluster_centers_.flatten()
    sorted_idx = np.argsort(centers)[::-1] 
    rank_map = {old: new + 1 for new, old in enumerate(sorted_idx)}
    
    return [f"Tier {rank_map[c]}" for c in clusters]

# 3. 체급별 순회 및 적용
final_list = []
for s in target_strata:
    temp = df_high[df_high['stratum'] == s].copy()
    if not temp.empty:
        temp['within_tier'] = get_4_tiers_within_stratum(temp)
        final_list.append(temp)

df_final = pd.concat(final_list)

# 4. 결과 요약
summary = df_final.groupby(['stratum', 'within_tier'])['total_reviews'].agg(['min', 'max', 'count']).reset_index()
summary.columns = ['체급(Strata)', '체급내 등급(Tier)', '최소 리뷰', '최대 리뷰', '게임 수']

print("="*85)
print("[체급별 4단계 분류] 각 목표 체급 내에서의 흥행 성과표")
print("="*85)
display(summary.sort_values(by=['체급(Strata)', '체급내 등급(Tier)']))

[체급별 4단계 분류] 각 목표 체급 내에서의 흥행 성과표


,체급(Strata),체급내 등급(Tier),최소 리뷰,최대 리뷰,게임 수
0,large_high,Tier 1,65046,153566,3
1,large_high,Tier 2,9093,22456,9
2,large_high,Tier 3,3663,8143,12
3,large_high,Tier 4,1191,2921,5
4,mid_high,Tier 1,3203,5090,2
5,mid_high,Tier 2,1076,1961,8
6,mid_high,Tier 3,562,767,7
7,mid_high,Tier 4,383,538,8
8,small_high,Tier 1,731,912,3
9,small_high,Tier 2,569,660,2
